# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/John-hcmus/flyrank-ML-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook documents the research question, lane choice, framing, starter data verification, and honest claim boundaries for the FlyRank ML Internship.

In [ ]:
# ⚡ SETUP TỰ ĐỘNG CHO GOOGLE COLAB VÀ MÔI TRƯỜNG CỤC BỘ
import os
import sys

# Kiểm tra nếu đang chạy trong môi trường Google Colab
if 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ:
    if not os.path.exists('data/raw/content_refresh_anonymized.csv') and not os.path.exists('../../data/raw/content_refresh_anonymized.csv'):
        print("⚡ Phát hiện Google Colab! Đang tự động clone repository để lấy file dữ liệu...")
        os.system('git clone https://github.com/John-hcmus/flyrank-ML-internship-starter.git /content/repo')
        if os.path.exists('/content/repo'):
            os.chdir('/content/repo')
            print(f"Đã cài đặt môi trường làm việc Colab tại: {os.getcwd()}")

print("✅ Môi trường đã sẵn sàng!")

## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Lý do lựa chọn (Why this lane):**
Trong quản trị nội dung website quy mô lớn, các bài viết theo thời gian thường bị giảm hiệu suất (content decay/decline) do thông tin trở nên lỗi thời, đối thủ cạnh tranh hoặc xu hướng tìm kiếm thay đổi. Hướng đi **Refresh / Content Opportunity Scoring** tập trung giải quyết bài toán cốt lõi: Xây dựng cơ chế chấm điểm và xếp hạng danh sách các bài viết cần được kiểm tra, cập nhật (refresh) hoặc tối ưu hóa theo thứ tự ưu tiên.

Thay vì huấn luyện một mô hình dự đoán trừu tượng, hướng đi này biến hàng nghìn số liệu tìm kiếm thành một **hàng đợi hành động thực tế (ranked review queue)** cho đội ngũ biên tập (content editors). Điều này giúp tối ưu hóa nguồn lực hạn chế của doanh nghiệp để bảo vệ và khôi phục những luồng lưu lượng truy cập (search traffic) có giá trị cao nhất.

In [ ]:
# Code check: Xác nhận định hướng Lane 2 kết nối trực tiếp với tập starter dataset
candidate_paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv'
]
data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

print(f"Lane selected: Refresh / Content Opportunity Scoring")
print(f"Target dataset path: '{data_path}' -> File exists: {data_path is not None}")

## 2. The question: decision, action, cost of a wrong call

**1. Unit of Analysis (Đối tượng phân tích / Grain):**
- Một bài viết / trang web cụ thể (`content_id`) thuộc về một khách hàng (`client_id`) trong cửa sổ thời gian 90 ngày (`trailing 90-day window`).

**2. Quyết định (Decision) & Đầu ra (Output):**
- **Quyết định:** "Trang web nào có nguy cơ suy giảm cao nhưng sở hữu lượng cầu đủ lớn cần được đội ngũ biên tập ưu tiên kiểm tra và làm mới (refresh) *trước tiên*?"
- **Đầu ra (Output):** Danh sách xếp hạng cơ hội (Ranked Review Queue) kèm theo điểm số tổng hợp (`final_refresh_score`) và các mã lý do hành động (`reason codes` như `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`).

**3. Người thực hiện & Hành động thực tế (Who acts & Action):**
- **Người thực hiện:** Đội ngũ Biên tập viên (Content Editors) hoặc Chuyên viên SEO (SEO Specialists).
- **Hành động:** Tiếp nhận Top-K bài viết từ danh sách ưu tiên, kiểm tra nội dung thực tế, cập nhật số liệu/thông tin mới, bổ sung đoạn văn thiếu sót, tối ưu lại thẻ tiêu đề/meta description hoặc điều chỉnh intent đáp ứng nhu cầu người dùng.

**4. Hậu quả của việc dự đoán sai (Cost of a wrong call):**
- **False Positive (Báo động giả):** Đưa một bài viết không hề suy giảm hoặc không cần sửa vào Top xếp hạng làm lãng phí thời gian, công sức của biên tập viên, đồng thời có nguy cơ gây xáo trộn điểm xếp hạng Google đang ổn định.
- **False Negative (Bỏ sót rủi ro):** Không phát hiện một bài viết quan trọng đang suy thoái thầm lặng, dẫn đến mất dần traffic và doanh thu vào tay đối thủ trước khi kịp can thiệp.

**5. Tại sao Dữ liệu / ML lại có ích (Why Data / ML helps):**
- Phân tích bằng tay hoặc quy tắc cứng (if-else heuristics) không thể xử lý hiệu quả quy mô hàng chục nghìn bài viết với đa tín hiệu chồng chéo (impressions, clicks, average position, GA4 sessions, engagement_rate, scroll_rate, content age, word count...). ML giúp tổng hợp đồng thời các tín hiệu này một cách khách quan, tối ưu thứ tự ưu tiên theo đúng năng lực xử lý (capacity) của biên tập viên.

In [ ]:
# Code check: Định nghĩa khung đánh giá Top-K (Decision Support framing)
top_k_capacity = 50
print(f"Decision Metric Focus: Precision@{top_k_capacity} (Chính xác trong Top {top_k_capacity} bài viết được khuyến nghị)")
print(f"Goal: Beat rule-based baseline in ranking high-priority content refresh candidates.")

## 3. Quick look at the data (2-3 real numbers)

Dưới đây là **3 con số thực tế** được trích xuất trực tiếp từ tập dữ liệu starter `content_refresh_anonymized.csv` (30,000 dòng x 44 cột, 32 khách hàng):

1. **16,262 trang (54.21%)** đang ở trạng thái suy giảm lượt hiển thị (`trend_direction == 'down'`).
2. **9,961 trang suy giảm** có lượng nhu cầu tìm kiếm cao (`impressions_90d >= 500`), chiếm **61.25%** tổng số trang suy giảm. Đây là nhóm đối tượng ưu tiên hàng đầu cần can thiệp.
3. **79,042,325 lượt hiển thị (impressions trong 90 ngày)** là tổng lưu lượng tìm kiếm đang chịu rủi ro suy thoái từ nhóm 9,961 trang nói trên.

In [ ]:
import os
import sys

# 1. Đường dẫn tự động tương thích mọi môi trường (Colab, VS Code, Jupyter Notebook)
candidate_paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv'
]
data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if not data_path:
    raise FileNotFoundError(f"Không tìm thấy file dataset. Các đường dẫn đã kiểm tra: {candidate_paths}")

print(f"Đã tìm thấy tập dữ liệu tại: '{data_path}'")

# 2. Tải và tính toán chỉ số (Hỗ trợ Pandas và tự động fallback sang CSV tích hợp nếu không có Pandas)
try:
    import pandas as pd
    df = pd.read_csv(data_path)
    total_rows = len(df)
    total_clients = df['client_id'].nunique()
    
    declining_mask = df['trend_direction'] == 'down'
    num_declining = declining_mask.sum()
    pct_declining = (num_declining / total_rows) * 100
    
    high_demand_declining = (declining_mask & (df['impressions_90d'] >= 500)).sum()
    pct_high_demand = (high_demand_declining / num_declining) * 100
    
    impressions_at_risk = df.loc[declining_mask & (df['impressions_90d'] >= 500), 'impressions_90d'].sum()

except Exception as e:
    import csv
    print(f"(Thông báo: Đang sử dụng thư viện csv chuẩn để xử lý dữ liệu - lý do: {e})")
    with open(data_path, mode='r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    total_rows = len(rows)
    total_clients = len(set(r['client_id'] for r in rows))
    
    declining_rows = [r for r in rows if r['trend_direction'] == 'down']
    num_declining = len(declining_rows)
    pct_declining = (num_declining / total_rows) * 100
    
    high_demand_list = [r for r in declining_rows if float(r['impressions_90d']) >= 500]
    high_demand_declining = len(high_demand_list)
    pct_high_demand = (high_demand_declining / num_declining) * 100
    
    impressions_at_risk = sum(float(r['impressions_90d']) for r in high_demand_list)

print("=" * 65)
print(f"Tổng số bài viết phân tích: {total_rows:,} bài (thuộc {total_clients} khách hàng)")
print(f"1. Số bài viết đang suy giảm (trend_direction == 'down'): {num_declining:,} ({pct_declining:.2f}%)")
print(f"2. Số bài viết suy giảm có nhu cầu cao (impressions_90d >= 500): {high_demand_declining:,} ({pct_high_demand:.2f}% nhóm suy giảm)")
print(f"3. Tổng lượt hiển thị 90 ngày chịu rủi ro suy thoái: {int(impressions_at_risk):,} lượt")
print("=" * 65)

## 4. Careful words: what I can and can't claim

**Những gì mô hình CÓ THỂ khẳng định (What the model CAN claim):**
- **Tín hiệu quan sát được (Observed signals):** Mô hình đưa ra điểm số và thứ tự ưu tiên dựa trên các số liệu thực tế đã diễn ra trong quá khứ (impressions, clicks, avg_position, GA4 sessions, content age...).
- **Hỗ trợ quyết định (Decision-support):** Mô hình cung cấp một danh sách đề xuất xếp hạng (ranked review queue) giúp biên tập viên phân bổ thời gian hợp lý hơn so với việc chọn ngẫu nhiên hay dùng quy tắc cảm tính.
- **Xu hướng tương quan (Directional associations):** Mô hình chỉ ra các yếu tố liên quan đến khả năng suy giảm lưu lượng truy cập.

**Những gì mô hình KHÔNG THỂ khẳng định (What the model CANNOT claim):**
- **Chứng minh quan hệ nguyên nhân - kết quả (Causal proof):** Mô hình *không thể* khẳng định việc cập nhật nội dung chắc chắn sẽ làm trang web khôi phục traffic (chưa qua thử nghiệm A/B test hoặc thiết kế thực nghiệm ngẫu nhiên).
- **Dự đoán thuật toán Google (Predicting Google algorithm):** Mô hình *không* giải mã hay dự đoán thuật toán bí mật của Google; mô hình chỉ phản ánh sự thay đổi chỉ số hiển thị quan sát được trên GSC/GA4.
- **Khẳng định trích dẫn AI (AI citations/rankings):** Chỉ số `ai_sessions_90d` chỉ ghi nhận lượt nhấp chuyển hướng (click-throughs) từ công cụ AI về website, *không* đo lường hay chứng minh bài viết có được mô hình AI trích dẫn hay không.

In [ ]:
# Code check: Tự kiểm tra tính an toàn của ngôn từ và dữ liệu
claim_types = {
    "Allowed": ["Observed signals", "Decision support ranking", "Directional associations"],
    "Forbidden": ["Causal proof of recovery", "Predicting Google algorithm", "AI citation counts"]
}
for category, claims in claim_types.items():
    print(f"{category} claims: {', '.join(claims)}")

## Self-check

Confirming each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.